In [4]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *
from models.lstm_xgb import LSTM_XGB

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "LSTM_XGB"


device: NVIDIA A30


In [5]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv("../CSV_Files/val.csv").to_numpy()
    test  = pd.read_csv("../CSV_Files/test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

In [6]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (LSTM_XGB) ===
  LSTM params: 4,744  breakdown: {'lstm': 4480, 'head': 264}
  [stage1] ep   1 | loss 1.7985 acc 0.257 | val loss 1.5767 acc 0.330
  [stage1] ep  10 | loss 0.4480 acc 0.802 | val loss 0.4221 acc 0.809
  [stage1] ep  20 | loss 0.3698 acc 0.830 | val loss 0.4500 acc 0.794
  [stage1] ep  30 | loss 0.3220 acc 0.857 | val loss 0.3706 acc 0.853
  [stage1] ep  40 | loss 0.3182 acc 0.858 | val loss 0.3618 acc 0.862
  [stage1] ep  50 | loss 0.2818 acc 0.879 | val loss 0.3136 acc 0.878
  [stage1] ep  60 | loss 0.2497 acc 0.896 | val loss 0.3382 acc 0.868
  [stage1] ep  70 | loss 0.1994 acc 0.921 | val loss 0.2838 acc 0.890
  TEST: acc=0.9169  P=0.9168  R=0.9169  F1=0.9166
  COMPUTE: stage1(LSTM)=38.5s  stage2(XGB)=0.9s  total=39.4s  inf=0.002ms/sample

=== Scenario 2 (LSTM_XGB) ===
  LSTM params: 4,744  breakdown: {'lstm': 4480, 'head': 264}
  [stage1] ep   1 | loss 1.8626 acc 0.229 | val loss 1.6141 acc 0.312
  [stage1] ep  10 | loss 1.0297 acc 0.561 | val loss 1.

In [7]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)


 scenario    model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 LSTM_XGB     7860    0.9169     0.9168  0.9169 0.9166      4744      39.40             0.0016        311.5
        2 LSTM_XGB     3912    0.8531     0.8549  0.8531 0.8531      4744      18.77             0.0017        167.2
        3 LSTM_XGB      778    0.4444     0.4514  0.4444 0.4463      4744       4.09             0.0017         82.8
        4 LSTM_XGB      394    0.4656     0.4685  0.4656 0.4630      4744       2.20             0.0016         82.8
        5 LSTM_XGB      236    0.5675     0.5709  0.5675 0.5678      4744       1.41             0.0015         82.8
        6 LSTM_XGB       75    0.3938     0.3729  0.3938 0.3771      4744       0.55             0.0013         82.8


In [8]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (LSTM_XGB, n_train=7860, acc=0.917)
[[168   2   0  30   0   0   0   0]
 [  0 196   0   0   0   0   0   4]
 [  0   0 195   0   0   5   0   0]
 [ 20   0   0 176   1   0   3   0]
 [  0   0   0   0 200   0   0   0]
 [  0   1   2   0   0 165   0  32]
 [  0   0   0   1   0   0 199   0]
 [  0  10   0   0   0  22   0 168]]

Scenario 2  (LSTM_XGB, n_train=3912, acc=0.853)
[[136   2   0  62   0   0   0   0]
 [  0 185   0   0   0   1   0  14]
 [  0   1 191   0   0   8   0   0]
 [ 55   0   0 139   0   0   6   0]
 [  0   0   0   0 200   0   0   0]
 [  0   4   0   0   0 149   0  47]
 [  0   0   0   2   0   0 198   0]
 [  0  14   1   0   0  18   0 167]]

Scenario 3  (LSTM_XGB, n_train=778, acc=0.444)
[[ 43  25  19  60   0  15  18  20]
 [ 26 103  13  22   1  10   9  16]
 [ 19  15  70  33   1  12  11  39]
 [ 30  16  17  71   1   8  23  34]
 [  0   0   2   0 188   9   1   0]
 [ 10   8  20   8   6 108  13  27]
 [ 20   9  18  35   0  14  60  44]
 [ 27   8  22  22   1  17  35  68]]

Scenario 4